In [13]:
from stormpy import export_to_drn
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('..')

from verimon.logger import setup_logging

setup_logging()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


[autoreload of verimon.verify failed: Traceback (most recent call last):
  File "/home/luko/.cache/pypoetry/virtualenvs/verimon-HSnaZ5_u-py3.11/lib/python3.11/site-packages/IPython/extensions/autoreload.py", line 276, in check
    superreload(m, reload, self.old_objects)
  File "/home/luko/.cache/pypoetry/virtualenvs/verimon-HSnaZ5_u-py3.11/lib/python3.11/site-packages/IPython/extensions/autoreload.py", line 475, in superreload
    module = reload(module)
             ^^^^^^^^^^^^^^
  File "/usr/lib/python3.11/importlib/__init__.py", line 169, in reload
    _bootstrap._exec(spec, module)
  File "<frozen importlib._bootstrap>", line 621, in _exec
  File "<frozen importlib._bootstrap_external>", line 940, in exec_module
  File "<frozen importlib._bootstrap>", line 241, in _call_with_frames_removed
  File "/home/luko/Documents/MDP-product/verimon/verify.py", line 14, in <module>
    from verimon.transformations import (
  File "/home/luko/Documents/MDP-product/verimon/transformations.py",

In [14]:
from verimon import loaders
from random import randrange
from math import sqrt


mc_sl_u_nxn = "../tests/snake_ladder/mc_u_nxn.pm"

n, ladders, snakes = loaders.random_snl_board(5**2)
# n, ladders, snakes = (25, {17: 19, 9: 15, 8: 15, 6: 10, 14: 21}, {23: 8, 22: 20, 8: 1, 18: 2, 12: 2})
print(n, ladders, snakes)

# Random snakes and ladders
# mc = loaders.load_snl_stormpy(mc_sl_u_nxn, n, ladders, snakes)

milton_snakes = {98: 76, 95: 75, 93: 73, 87: 24, 64: 60, 62: 19, 55: 53, 49: 11, 47: 26, 16: 6}
milton_ladders = {1: 38, 4: 14, 9: 31, 28: 64, 40: 42, 36: 44, 51: 67, 71: 91, 80: 100}
mc = loaders.load_snl_stormpy(mc_sl_u_nxn, n := 10**2, ladders:=milton_ladders, snakes:=milton_snakes)

25 {4: 10, 19: 22, 3: 14, 17: 20, 5: 12} {14: 12, 22: 20, 13: 4, 15: 4, 20: 2}


In [15]:
from stormvogel.mapping import stormpy_to_stormvogel
from stormvogel.show import show
import stormvogel

stormvogel.communication_server.enable_server = False

mc_sv = stormpy_to_stormvogel(mc)
if mc_sv is None:
    raise Exception("boom")
loaders._add_valuation_to_sv_labels(mc, mc_sv)
show(mc_sv)

In [16]:
%matplotlib notebook
from verimon.draw import animate_player_movement
import math
from IPython.display import HTML

player_path = [(0, [])]

goal_squares = [next(int(l[5:-1]) for l in state.labels if l.startswith("[pos")) 
                for state in mc_sv.states.values() 
                if "good" in state.labels]

animation = animate_player_movement(int(math.sqrt(n)), snakes, ladders, goal_squares, player_path)
HTML(animation.to_jshtml())

<IPython.core.display.Javascript object>

In [23]:
from aalpy import run_Lstar, Dfa
from verimon.MonitorLearning import FilteringSUL, VerimonEqOracle

setup_logging()


threshold = 0.4
slack = 0.1
horizon = 12
relative_error = 0.1
spec = 'P>0.5 [F<3 "good" ]'

alphabet = ["init", "normal", "snake", "ladder"]

filtering_sul = FilteringSUL(
    mc, 
    "init", 
    alphabet, 
    spec, 
    threshold, 
    horizon
)
eq_oracle = VerimonEqOracle(
    alphabet,
    filtering_sul,
    mc,
    threshold,
    slack,
    horizon,
    spec,
    'good',
    relative_error
    
)
learned_monitor: Dfa = run_Lstar(
    alphabet,
    filtering_sul,
    eq_oracle,
    automaton_type="dfa",
    print_level=2,
) #type: ignore

Hypothesis 1: 1 states.
Visualization started in the background thread.
DEBUG:2024-11-11 14:21:49,109 - (0.00s) - MonitorLearning.py - Finding false negative probability 
DEBUG:2024-11-11 14:21:49,110 - (0.00s) - verify.py - Building model 
DEBUG:2024-11-11 14:21:49,111 - (0.00s) - verify.py - Unrolling done 
DEBUG:2024-11-11 14:21:49,113 - (0.00s) - verify.py - Pruning done 
INFO:2024-11-11 14:21:49,116 - (0.00s) - generator.py - New good states become: [81, 93] 
DEBUG:2024-11-11 14:21:49,118 - (0.00s) - verify.py - Apply spec done 
DEBUG:2024-11-11 14:21:49,159 - (0.04s) - verify.py - creating product done 
DEBUG:2024-11-11 14:21:49,159 - (0.00s) - verify.py - Creating trace 
Model saved to models/model1.pdf.
> progress 20.0%, elapsed 4 s, estimated 20 s, iters = {MDP: 89}
> progress 40.002%, elapsed 7 s, estimated 17 s, iters = {MDP: 229}
> progress 40.96%, elapsed 10 s, estimated 24 s, iters = {MDP: 314}
INFO:2024-11-11 14:21:59,862 - (10.70s) - generator.py - --------------------


In [24]:
from verimon.MonitorLearning import aalpy_dfa_to_stormvogel
from verimon.transformations import simulator_unroll, prune_monitor
from verimon.algs import complement_model

mon_cycl = aalpy_dfa_to_stormvogel(learned_monitor)
# show(mon_cycl)
# complement_model(mon_cycl, "accepting")
mon = simulator_unroll(mon_cycl, horizon)
prune_monitor(mon)
print(len(mon.states))
show(mon)

47


In [25]:
from verimon.MonitorLearning import aalpy_dfa_to_stormvogel
from verimon.verify import *

mon_cycl = aalpy_dfa_to_stormvogel(learned_monitor)
export_to_drn(stormvogel_to_stormpy(mon_cycl), "monitor.drn")
result_goal, trace, assignment, product = false_positive(mc, mon_cycl, horizon, options = {"good_spec": spec})

DEBUG:2024-11-11 14:24:02,629 - (66.71s) - verify.py - Building model 
Write to file monitor.drn.
DEBUG:2024-11-11 14:24:02,631 - (0.00s) - verify.py - Unrolling done 
DEBUG:2024-11-11 14:24:02,652 - (0.02s) - verify.py - Pruning done 
INFO:2024-11-11 14:24:02,655 - (0.00s) - generator.py - New good states become: [81, 93] 
DEBUG:2024-11-11 14:24:02,655 - (0.00s) - verify.py - Apply spec done 
DEBUG:2024-11-11 14:24:02,670 - (0.01s) - verify.py - creating product done 


DEBUG:2024-11-11 14:24:02,670 - (0.00s) - verify.py - Creating trace 
> progress 32.938%, elapsed 3 s, estimated 9 s, iters = {MDP: 154}, opt = 0.5004
INFO:2024-11-11 14:24:06,786 - (4.12s) - generator.py - --------------------
Synthesis summary:
optimality objective: Pmax=? [F "stop"] [eps = 0.1]

method: AR, synthesis time: 4.04 s
number of holes: 14, family size: 1e8, quotient: 274 states / 1116 actions
explored: 100 %
MDP stats: avg MDP size: 77, iterations: 193

feasible: yes
--------------------
 
INFO:2024-11-11 14:24:06,787 - (0.00s) - verify.py - Found trace: ['ladder', 'normal', 'normal', 'normal', 'ladder', 'normal', 'ladder', 'normal', 'normal', 'normal', 'normal'] 
INFO:2024-11-11 14:24:06,791 - (0.00s) - verify.py - Goal probability counterexample: 0.45454545454513406 


In [21]:
%matplotlib
from verimon.draw import animate_player_movement
import math
from IPython.display import HTML

# player_path = [(0, [])]
poss = product.simulate_paynt_assignment(assignment, 100000)
player_path = poss

goal_squares = [int(str(state.valuations)[5:-1]) 
                for state in product.mc.states 
                if "good" in state.labels]

animation = animate_player_movement(int(math.sqrt(n)), snakes, ladders, goal_squares, player_path)
HTML(animation.to_jshtml())

Using matplotlib backend: notebook
INFO:2024-11-11 14:17:36,431 - (4.65s) - generator.py - s2, obs=2, labels=init 


INFO:2024-11-11 14:17:38,194 - (1.76s) - generator.py - --[2, ladder]-->	s6, val=[pos=1], labels=
--[3, normal]-->	s10, val=[pos=38], labels=
--[4, normal]-->	s16, val=[pos=42], labels=
--[5, normal]-->	s33, val=[pos=46], labels=
--[6, ladder]-->	s56, val=[pos=51], labels=
--[7, normal]-->	s88, val=[pos=67], labels=
--[8, ladder]-->	s109, val=[pos=71], labels=
--[9, normal]-->	s129, val=[pos=91], labels=
--[10, normal]-->	s165, val=[pos=97], labels=
--[11, normal]-->	s226, val=[pos=100], labels=
--[13, normal]-->	s262, val=[pos=100], labels=
--[15, end]-->	s0, val=[pos=-1], labels=goal 
INFO:2024-11-11 14:17:38,195 - (0.00s) - generator.py - it took 3261 tries until the goal was reached 


<IPython.core.display.Javascript object>